# Jobsheet 5

## Setup Environment

In [ ]:
%pip install -q transformers

In [ ]:
import os, time, random, json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision

def set_seed(seed=42):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CEPAT = DEVICE == "cpu"     # tanpa GPU: jalankan versi diperkecil
CEPAT = DEVICE == "cpu"   # tanpa GPU: jalankan versi diperkecil
print("PyTorch", torch.__version__, "| torchvision", torchvision.__version__)
print("Perangkat:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU saja")

CFG = dict(
    IMG     = 128 if CEPAT else 256,    #ukuran sisi citar (pixel)
    N_TRAIN = 320 if CEPAT else 2000,   # jumlah citra latih (dari daftar traincal resmi)
    N_TEST  = 100 if CEPAT else 500,    # jumlah citra uji (dari daftar testcal resmi)
    BATCH   = 16,
    EP_UNET = 6 if CEPAT else 15,
    EP_DLV3 = 2 if CEPAT else 6,
    EP_SEGF = 3 if CEPAT else 6,
    BASE_CH = 16 if CEPAT else 32       # lebar channel unet
)

print("Mode:", "CEPAT (CPU)" if CEPAT else "PENUH (GPU)", CFG)
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)
WAKTU = {}   # catatan waktu tiap bagian

## Praktikum D1: Mengenal Data Segmentasi Oxford-IIIT Pet

 nilai 1 untuk hewan, 2 untuk latar, dan 3 untuk batas yang ragu-ragu.Kita ubah trimap menjadi masker biner (hewan = 1, latar = 0) dan masker valid yang bernilai 0 pada piksel batas.

In [ ]:
import os
import tarfile
import time
import urllib.request
import random
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch

# === D1.1 Unduh subset Oxford-IIIT Pet langsung dari server resmi VGG ===
URL_IMG = "https://thor.robots.ox.ac.uk/pets/images.tar.gz"
URL_ANN = "https://thor.robots.ox.ac.uk/pets/annotations.tar.gz"

def ambil_subset_pet(root="data/pet_subset", n_train=480, n_test=120):
    os.makedirs(f"{root}/images", exist_ok=True)
    os.makedirs(f"{root}/trimaps", exist_ok=True)

    ann = f"{root}/annotations.tar.gz"
    if not os.path.exists(ann):
        urllib.request.urlretrieve(URL_ANN, ann)

    split, trimaps = {}, {}
    with tarfile.open(ann, "r:gz") as t:
        for m in t:
            if m.name.endswith("annotations/trainval.txt") or m.name.endswith("annotations/test.txt"):
                s = "train" if "trainval" in m.name else "test"
                for line in t.extractfile(m).read().decode().splitlines():
                    if line.strip():
                        split[line.split()[0]] = s
            elif "/trimaps/" in m.name and m.name.endswith(".png") and "._" not in m.name:
                trimaps[os.path.basename(m.name)[:-4]] = t.extractfile(m).read()

    def hitung():
        ada = [f[:-4] for f in os.listdir(f"{root}/images") if f.endswith(".jpg")]
        return (
            sorted(i for i in ada if split.get(i) == "train"),
            sorted(i for i in ada if split.get(i) == "test")
        )

    tr, te = hitung()
    if len(tr) >= n_train and len(te) >= n_test:
        print("Subset sudah tersedia di", root)
        return tr[:n_train], te[:n_test]

    with urllib.request.urlopen(URL_IMG) as resp, tarfile.open(fileobj=resp, mode="r|gz") as t:
        for m in t:
            nama = os.path.basename(m.name)
            if not nama.endswith(".jpg"):
                continue
            iid = nama[:-4]
            s = split.get(iid)
            if iid not in trimaps or s is None:
                continue
            if (s == "train" and len(tr) >= n_train) or (s == "test" and len(te) >= n_test):
                continue

            with open(f"{root}/images/{nama}", "wb") as f:
                f.write(t.extractfile(m).read())
            with open(f"{root}/trimaps/{iid}.png", "wb") as f:
                f.write(trimaps[iid])

            (tr if s == "train" else te).append(iid)
            if len(tr) >= n_train and len(te) >= n_test:
                break

    tr, te = hitung()
    return tr[:n_train], te[:n_test]

t0 = time.time()
ROOT = "data/pet_subset"
ID_TRAIN, ID_TEST = ambil_subset_pet(ROOT, CFG["N_TRAIN"], CFG["N_TEST"])
print(f"Citra latih: {len(ID_TRAIN)} | citra uji: {len(ID_TEST)} | waktu {time.time()-t0:.1f} detik")
print("Jumlah ras pada data latih:", len({i.rsplit('_', 1)[0] for i in ID_TRAIN}))

# === D1.2 Dataset PyTorch: citra RGB + masker biner + masker valid ===
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def muat_pasangan(iid, size):
    img = Image.open(f"{ROOT}/images/{iid}.jpg").convert("RGB").resize((size, size), Image.BILINEAR)
    tri = Image.open(f"{ROOT}/trimaps/{iid}.png").resize((size, size), Image.NEAREST)
    return np.array(img), np.array(tri)

class PetSegDataset(torch.utils.data.Dataset):
    # Semua data dimuat sekali ke memori dalam bentuk uint8 agar pelatihan cepat
    def __init__(self, ids, size, augment=False):
        pairs = [muat_pasangan(i, size) for i in ids]
        self.img = torch.from_numpy(np.stack([p[0] for p in pairs])).permute(0, 3, 1, 2)  # (N,3,H,W) uint8
        self.tri = torch.from_numpy(np.stack([p[1] for p in pairs]))                      # (N,H,W)   uint8
        self.augment = augment

    def __len__(self):
        return len(self.img)

    def __getitem__(self, i):
        x = self.img[i].float() / 255.0
        tri = self.tri[i]
        y = (tri == 1).float().unsqueeze(0)       # 1 = hewan, 0 = latar
        valid = (tri != 3).float().unsqueeze(0)   # 0 pada piksel batas (diabaikan)

        if self.augment and random.random() < 0.5:    # augment: flip horizontal
            x, y, valid = x.flip(-1), y.flip(-1), valid.flip(-1)

        return (x - MEAN) / STD, y, valid

t0 = time.time()
ds_train = PetSegDataset(ID_TRAIN, CFG["IMG"], augment=True)
ds_test  = PetSegDataset(ID_TEST,  CFG["IMG"], augment=False)  # Diperbaiki: augment=False untuk data uji

dl_train = torch.utils.data.DataLoader(ds_train, batch_size=CFG["BATCH"], shuffle=True)
dl_test  = torch.utils.data.DataLoader(ds_test,  batch_size=CFG["BATCH"], shuffle=False)

x, y, v = ds_train[0]
print("Bentuk x:", tuple(x.shape), "| y:", tuple(y.shape), "| valid:", tuple(v.shape))
tri_all = ds_train.tri
print("Proporsi piksel  hewan: %.3f | latar: %.3f | batas: %.3f" % tuple(((tri_all == k).float().mean().item()) for k in (1, 2, 3)))
WAKTU["D1 data"] = time.time() - t0

# === D1.3 Visualisasi: citra, trimap, dan masker biner ===
def denorm(x):
    return (x * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

fig, ax = plt.subplots(3, 4, figsize=(12, 9))
for k in range(4):
    x, y, v = ds_test[k]
    ax[0, k].imshow(denorm(x))
    ax[0, k].set_title(ID_TEST[k], fontsize=9)

    ax[1, k].imshow(ds_test.tri[k], cmap="viridis", vmin=1, vmax=3)
    ax[1, k].set_title("trimap (1 hewan, 2 latar, 3 batas)", fontsize=8)

    ax[2, k].imshow(y[0], cmap="gray")
    ax[2, k].set_title("masker biner", fontsize=9)

for a in ax.ravel():
    a.axis("off")

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/js05_d1_contoh_data.png", dpi=120)
plt.show()